# Benchmark 2.0 — Part C: Merge & Analisis Final

Notebook ini **menggabungkan hasil** dari:
- **Part A** (Google Colab): MobileNetV3Large, MobileNetV2, EfficientNetB0
- **Part B** (Kaggle): EfficientNetV2S, ConvNeXtTiny, ResNet50

Lalu menghasilkan semua tabel, visualisasi, dan uji statistik untuk jurnal.

### Yang Dibutuhkan
Upload file-file berikut (dari ZIP Part A & B):
```
all_results_partA.pkl    ← dari Part A
all_results_partB.pkl    ← dari Part B
deployment_partA.pkl     ← dari Part A
deployment_partB.pkl     ← dari Part B
best_<model_terbaik>.keras  ← untuk Grad-CAM (opsional)
```

In [ ]:
# ========================================
# Setup & Upload File Hasil
# ========================================
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from scipy import stats as scipy_stats
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 150
sns.set_style("whitegrid")

MERGE_DIR = Path("/content/benchmark2_merged")
MERGE_DIR.mkdir(parents=True, exist_ok=True)

print("Upload 2 file yang diperlukan:")
print("  1. all_results.pkl  dari Part A (Colab)")
print("  2. all_results.pkl  dari Part B (Kaggle)")
print("  3. deployment.pkl   dari Part A (Colab)")
print("  4. deployment.pkl   dari Part B (Kaggle)")
print("\nGunakan cell berikut untuk upload.")

In [ ]:
# ========================================
# Upload File dari Colab & Kaggle
# ========================================
try:
    from google.colab import files
    print("Upload all_results_partA.pkl (dari Colab)...")
    up = files.upload()
    print("Upload all_results_partB.pkl (dari Kaggle)...")
    up2 = files.upload()
    print("Upload deployment_partA.pkl (dari Colab)...")
    up3 = files.upload()
    print("Upload deployment_partB.pkl (dari Kaggle)...")
    up4 = files.upload()
    print("Semua file terupload.")
except ImportError:
    print("Bukan di Colab — letakkan file pkl secara manual di /content/")

# ── Rename ──────────────────────────────────────────────────────────────────
# Setelah upload, rename sesuai konvensi:
#   all_results_partA.pkl, all_results_partB.pkl
#   deployment_partA.pkl,  deployment_partB.pkl
# Sesuaikan path jika berbeda
PARTA_RESULTS   = "/content/all_results_partA.pkl"
PARTB_RESULTS   = "/content/all_results_partB.pkl"
PARTA_DEPLOY    = "/content/deployment_partA.pkl"
PARTB_DEPLOY    = "/content/deployment_partB.pkl"

with open(PARTA_RESULTS, "rb") as f:
    results_a = pickle.load(f)
with open(PARTB_RESULTS, "rb") as f:
    results_b = pickle.load(f)
with open(PARTA_DEPLOY, "rb") as f:
    deploy_a = pickle.load(f)
with open(PARTB_DEPLOY, "rb") as f:
    deploy_b = pickle.load(f)

# Gabungkan
all_results = {**results_a, **results_b}
deployment  = {**deploy_a,  **deploy_b}

# Ambil class_names dari salah satu hasil
sample_key  = list(all_results.keys())[0]
class_names = all_results[sample_key]["class_names"]
NUM_CLASSES = len(class_names)

MODEL_ORDER = [
    "MobileNetV3Large", "MobileNetV2", "EfficientNetB0",
    "EfficientNetV2S", "ConvNeXtTiny", "ResNet50",
]
# Filter hanya yang ada
MODEL_ORDER = [m for m in MODEL_ORDER if m in all_results]

print(f"Model tersedia: {MODEL_ORDER}")
print(f"Kelas: {class_names}")

In [ ]:
# ========================================
# Tabel Master Hasil Gabungan
# ========================================
master_rows = []
for name in MODEL_ORDER:
    r  = all_results[name]
    p2 = r["phase2"]
    dep = deployment.get(name, {})
    master_rows.append({
        "Model"         : name,
        "Accuracy (%)"  : f"{np.mean(p2['acc'])*100:.2f} ± {np.std(p2['acc'])*100:.2f}",
        "Macro F1"      : f"{np.mean(p2['f1']):.4f} ± {np.std(p2['f1']):.4f}",
        "Macro Precision": f"{np.mean(p2['prec']):.4f} ± {np.std(p2['prec']):.4f}",
        "Macro Recall"  : f"{np.mean(p2['rec']):.4f} ± {np.std(p2['rec']):.4f}",
        "Params (M)"    : f"{dep.get('params_m', 0):.2f}",
        "TFLite (MB)"   : f"{dep.get('tflite_mb', 0):.2f}",
        "Latency (ms)"  : f"{dep.get('latency_ms', 0):.2f}",
    })

master_df = pd.DataFrame(master_rows)
print("=" * 100)
print("  TABEL MASTER — BENCHMARK 2.0 (6 ARSITEKTUR, 5-FOLD CV, 2-PHASE)")
print("=" * 100)
display(master_df)
master_df.to_csv(MERGE_DIR / "benchmark2_master_table.csv", index=False)

# Per-class metrics
pc_rows = []
for name in MODEL_ORDER:
    p2  = all_results[name]["phase2"]
    mp  = np.mean(p2["pc_p"], axis=0)
    sp  = np.std(p2["pc_p"], axis=0)
    mr_ = np.mean(p2["pc_r"], axis=0)
    sr  = np.std(p2["pc_r"], axis=0)
    mf  = np.mean(p2["pc_f1"], axis=0)
    sf  = np.std(p2["pc_f1"], axis=0)
    for ci, cn in enumerate(class_names):
        pc_rows.append({
            "Model": name, "Kelas": cn,
            "Precision": f"{mp[ci]:.4f} ± {sp[ci]:.4f}",
            "Recall"   : f"{mr_[ci]:.4f} ± {sr[ci]:.4f}",
            "F1-Score" : f"{mf[ci]:.4f} ± {sf[ci]:.4f}",
        })
pc_df = pd.DataFrame(pc_rows)
display(pc_df)
pc_df.to_csv(MERGE_DIR / "benchmark2_per_class_metrics.csv", index=False)

In [ ]:
# ========================================
# Visualisasi Gabungan
# ========================================
colors = ["#2b5c8f", "#3a7bbf", "#4e94d4", "#81b29a", "#a7c957", "#e07a5f"]

# 1. Confusion Matrix Agregat (Phase 2)
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for idx, name in enumerate(MODEL_ORDER):
    p   = all_results[name]["phase2"]
    yt  = np.concatenate(p["y_trues"])
    yp_ = np.concatenate(p["y_preds"])
    cm  = confusion_matrix(yt, yp_, labels=range(NUM_CLASSES))
    acc = accuracy_score(yt, yp_)
    f1  = f1_score(yt, yp_, average="macro", zero_division=0)
    ax  = axes[idx // 3][idx % 3]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{name}\nAcc: {acc*100:.1f}% | F1: {f1:.3f}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
plt.suptitle("Confusion Matrix Agregat (Phase 2 — Fine-Tuning, 5-Fold CV)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(MERGE_DIR / "confusion_matrices_phase2.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. Phase 1 vs Phase 2
p1m = [np.mean(all_results[n]["phase1"]["acc"])*100 for n in MODEL_ORDER]
p2m = [np.mean(all_results[n]["phase2"]["acc"])*100 for n in MODEL_ORDER]
p1s = [np.std(all_results[n]["phase1"]["acc"])*100  for n in MODEL_ORDER]
p2s = [np.std(all_results[n]["phase2"]["acc"])*100  for n in MODEL_ORDER]
x   = np.arange(len(MODEL_ORDER)); w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - w/2, p1m, w, yerr=p1s, label="Phase 1 (Frozen)", color="#2b5c8f", alpha=0.8, capsize=4)
b2 = ax.bar(x + w/2, p2m, w, yerr=p2s, label="Phase 2 (Fine-Tuned)", color="#e07a5f", alpha=0.8, capsize=4)
ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 110)
ax.set_title("Phase 1 vs Phase 2 — Mean ± Std (5-Fold CV)", fontweight="bold")
ax.legend()
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 2,
                f"{h:.1f}%", ha="center", fontsize=8, fontweight="bold")
plt.tight_layout()
plt.savefig(MERGE_DIR / "phase_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

# 3. Box plot variabilitas fold
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
p2_accs = [all_results[n]["phase2"]["acc"] for n in MODEL_ORDER]
p2_f1s  = [all_results[n]["phase2"]["f1"]  for n in MODEL_ORDER]
labels  = [n.replace("Net", "\nNet") for n in MODEL_ORDER]

for ax, data, ylabel, title in [
    (axes[0], p2_accs, "Accuracy",  "Distribusi Accuracy per Fold (Phase 2)"),
    (axes[1], p2_f1s,  "Macro F1",  "Distribusi F1 per Fold (Phase 2)"),
]:
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")

plt.tight_layout()
plt.savefig(MERGE_DIR / "fold_variability.png", dpi=200, bbox_inches="tight")
plt.show()

# 4. Tradeoff chart
dl = [n for n in MODEL_ORDER if n in deployment]
accs_vals = [np.mean(all_results[n]["phase2"]["acc"])*100 for n in dl]
sizes_vals = [deployment[n]["tflite_mb"]  for n in dl]
lats_vals  = [deployment[n]["latency_ms"] for n in dl]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, vals, title, color in [
    (axes[0], accs_vals,  "Test Accuracy (%)",     "#2b5c8f"),
    (axes[1], sizes_vals, "Ukuran TFLite (MB)",    "#e07a5f"),
    (axes[2], lats_vals,  "Latensi CPU (ms)",       "#81b29a"),
]:
    ax.bar(dl, vals, color=color, alpha=0.85)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*0.02, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")
plt.suptitle("Tradeoff: Akurasi vs Efisiensi Deployment", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(MERGE_DIR / "tradeoff_chart.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ========================================
# McNemar's Test (Uji Signifikansi)
# ========================================
agg_preds  = {}
y_true_global = None

for name in MODEL_ORDER:
    p  = all_results[name]["phase2"]
    yt = np.concatenate(p["y_trues"])
    yp = np.concatenate(p["y_preds"])
    agg_preds[name] = yp
    if y_true_global is None:
        y_true_global = yt

rows = []
for i in range(len(MODEL_ORDER)):
    for j in range(i + 1, len(MODEL_ORDER)):
        a, b = MODEL_ORDER[i], MODEL_ORDER[j]
        ac  = agg_preds[a] == y_true_global
        bc  = agg_preds[b] == y_true_global
        n01 = int(np.sum(ac & ~bc))
        n10 = int(np.sum(~ac & bc))
        if n01 + n10 > 0:
            chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
            pval = 1 - scipy_stats.chi2.cdf(chi2, 1)
        else:
            chi2, pval = 0.0, 1.0
        rows.append({
            "Model A": a, "Model B": b,
            "A>B": n01, "B>A": n10,
            "chi2": f"{chi2:.4f}", "p-value": f"{pval:.4f}",
            "Signifikan (α=0.05)": "Ya ✓" if pval < 0.05 else "Tidak",
        })

mcn_df = pd.DataFrame(rows)
print("McNemar's Test — Phase 2 (Fine-Tuning)\n")
display(mcn_df)
mcn_df.to_csv(MERGE_DIR / "mcnemar_test_results.csv", index=False)

In [ ]:
# ========================================
# Grad-CAM (Upload best_*.keras model terbaik)
# ========================================
# Temukan model terbaik
best_name = max(MODEL_ORDER, key=lambda n: np.mean(all_results[n]["phase2"]["acc"]))
print(f"Model terbaik: {best_name}")
print(f"  Mean Acc: {np.mean(all_results[best_name]['phase2']['acc'])*100:.2f}%")

# Upload model terbaik
try:
    from google.colab import files
    print(f"\nUpload file: best_{best_name}.keras")
    files.upload()
except ImportError:
    print("Letakkan file model di /content/ secara manual.")

MODEL_PATH = f"/content/best_{best_name}.keras"
if not os.path.exists(MODEL_PATH):
    print(f"File {MODEL_PATH} tidak ditemukan. Skip Grad-CAM.")
else:
    # Preprocess function lookup
    PFN_MAP = {
        "MobileNetV3Large": tf.keras.applications.mobilenet_v3.preprocess_input,
        "MobileNetV2"     : tf.keras.applications.mobilenet_v2.preprocess_input,
        "EfficientNetB0"  : tf.keras.applications.efficientnet.preprocess_input,
        "EfficientNetV2S" : tf.keras.applications.efficientnet_v2.preprocess_input,
        "ConvNeXtTiny"    : tf.keras.applications.convnext.preprocess_input,
        "ResNet50"        : tf.keras.applications.resnet50.preprocess_input,
    }
    pfn = PFN_MAP[best_name]
    best_model = tf.keras.models.load_model(MODEL_PATH)

    # Load sampel gambar dari results (gunakan y_trues/y_preds untuk index)
    # Kita re-download dataset minimal untuk Grad-CAM
    from roboflow import Roboflow
    from tensorflow.keras.utils import load_img, img_to_array

    rf = Roboflow(api_key="oSqbdWgbnJkiVfOWl18o")
    dataset = rf.workspace("valll-rtclj").project("cattle-breeds-zrtzt").version(2).download("folder")
    DATASET_DIR = dataset.location
    IMAGE_SIZE = (224, 224)

    sample_imgs, sample_labels = [], []
    for split in ["train", "valid", "test"]:
        sdir = os.path.join(DATASET_DIR, split)
        if not os.path.exists(sdir): continue
        for cls in sorted(os.listdir(sdir)):
            cdir = os.path.join(sdir, cls)
            if not os.path.isdir(cdir): continue
            imgs = [f for f in sorted(os.listdir(cdir))
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            if imgs:
                sample_imgs.append(os.path.join(cdir, imgs[0]))
                sample_labels.append(cls)
                break  # 1 per class

    label_to_idx = {n: i for i, n in enumerate(class_names)}

    def find_last_conv(model):
        for layer in reversed(model.layers):
            if len(layer.output.shape) == 4:
                return layer.name
        return None

    def gradcam(model, img_batch, target_cls, layer_name):
        from tensorflow.keras.models import Model as KModel
        grad_m = KModel(inputs=model.input,
                        outputs=[model.get_layer(layer_name).output, model.output])
        with tf.GradientTape() as tape:
            conv_out, preds = grad_m(img_batch)
            loss = preds[:, target_cls]
        grads   = tape.gradient(loss, conv_out)
        weights = tf.reduce_mean(grads, axis=(1, 2))
        cam     = tf.reduce_sum(conv_out * weights[:, tf.newaxis, tf.newaxis, :], axis=-1)
        cam     = tf.nn.relu(cam)
        cam     = cam / (tf.reduce_max(cam) + 1e-8)
        return cam.numpy()[0]

    target_layer = find_last_conv(best_model)
    n_cls = len(sample_imgs)
    fig, axes = plt.subplots(2, n_cls, figsize=(3 * n_cls, 6))

    for ci, (path, cls) in enumerate(zip(sample_imgs, sample_labels)):
        raw  = img_to_array(load_img(path, target_size=IMAGE_SIZE))[np.newaxis]
        proc = pfn(raw.copy())
        pred = best_model.predict(proc, verbose=0)
        pc   = int(np.argmax(pred[0]))
        conf = pred[0][pc]
        hm   = gradcam(best_model, proc, pc, target_layer)
        hm_r = tf.image.resize(hm[..., np.newaxis], IMAGE_SIZE).numpy()[:, :, 0]

        axes[0, ci].imshow(raw[0].astype(np.uint8)); axes[0, ci].axis("off")
        axes[0, ci].set_title(cls, fontweight="bold", fontsize=10)
        axes[1, ci].imshow(raw[0].astype(np.uint8))
        axes[1, ci].imshow(hm_r, alpha=0.5, cmap="jet"); axes[1, ci].axis("off")
        axes[1, ci].set_title(f"{class_names[pc]} ({conf:.0%})", fontsize=9)

    axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
    axes[1, 0].set_ylabel("Grad-CAM", fontsize=12, fontweight="bold")
    plt.suptitle(f"Grad-CAM — {best_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(MERGE_DIR / "gradcam_examples.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# ========================================
# Export Final ZIP
# ========================================
import shutil
zip_path = shutil.make_archive("/content/benchmark2_final", "zip", "/content", "benchmark2_merged")
print(f"ZIP: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("File tersedia di:", zip_path)